# 1x1 SUMO-RL Results Report

**Project:** Pedestrian-safety-aware traffic signal control  
**Scenario:** 1x1 single intersection  
**Evaluation:** 5 episodes, 3,600 simulation seconds each  
**Main question:** can a learned controller reduce delay while respecting traffic-signal and pedestrian-safety constraints?

This standalone report uses saved result files only. It does not rerun training or evaluation.


## Experimental Setup

| Setting | Value |
|---|---|
| SUMO config | `sumo_rl/nets/single-intersection/single_intersection.sumocfg` |
| Controlled TLS | `c` |
| State dimension | 23 |
| Action dimension | 4 green actions |
| Evaluation episodes | 5 |
| Simulation length | 3,600 s per episode |
| Original PPO training | 100 episodes |
| Tuned PPO training | 100 episodes |
| DQN-AR training | 100 episodes |

The main PPO row is the original final PPO model because it achieved the best all-agent final evaluation reward. The tuned PPO is included as a hyperparameter comparison.


## Table 1: 1x1 Agent Performance

| Agent | Avg Wait (s) | Avg Ped Wait (s) | Avg Queue | Violation Rate | Total Reward |
|---|---:|---:|---:|---:|---:|
| Original PPO | 60.28 | 0.0025 | 4.271 | 0.0% | -217,026 |
| Tuned PPO | 60.78 | 0.0003 | 4.453 | 0.0% | -218,798 |
| DQN-AR | 65.85 | 0.0058 | 4.468 | 0.0% | -237,073 |
| SPRe+ (DQN policy) | 65.85 | 0.0058 | 4.468 | 0.0% | -237,073 |
| SOTL (kappa=5) | 71.64 | 0.0009 | 4.778 | 18.6% | -291,292 |
| Max Pressure | 55.63 | 0.4015 | 4.036 | 53.5% | -297,984 |
| Fixed-Time (30s cycle) | 107.58 | 0.0025 | 5.606 | 57.2% | -490,231 |

**Source:** `results/eval_1x1_all_agents_5ep_real_metrics_20260606_020244/` and `results/eval_1x1_tunedppo_all_agents_5ep_ppo_tuned_20260606_122740/`.


## Figure 1: Metric Comparison

The plots compare delay, queue length, safety violations, and total reward across all evaluated agents.


In [ ]:
from IPython.display import Image, display
display(Image(filename=r'C:/Users/hong/Documents/GitHub/RL_final_project/RL_final_project/final_results/1x1_metrics_comparison.png'))


## Figure 2: Efficiency vs Safety

Lower-left is better: lower vehicle waiting time and lower violation rate. Max Pressure is efficient but unsafe under the constraint checker, while PPO and DQN-based methods keep violation rate at zero.


In [ ]:
from IPython.display import Image, display
display(Image(filename=r'C:/Users/hong/Documents/GitHub/RL_final_project/RL_final_project/final_results/1x1_efficiency_vs_safety.png'))


## PPO Hyperparameter Comparison

| Metric | Original PPO | Tuned PPO | Preferred Direction |
|---|---:|---:|---|
| Final eval total reward | -2.17e+05 | -2.188e+05 | Higher / less negative |
| Final eval avg wait | 60.28 | 60.78 | Lower |
| Final eval avg queue | 4.271 | 4.453 | Lower |
| Best eval reward during training | -2,153 | -2,106 | Higher / less negative |
| Final train explained variance | 0.1177 | 0.6271 | Higher |
| Final train value loss | 167.1 | 44.49 | Lower |
| Final train entropy | 0.01134 | 0.007581 | Context-dependent |
| Final train clip fraction | 0.00265 | 0.004167 | Moderate |

The tuned PPO improved critic diagnostics substantially. Its final training explained variance increased from `0.118` to `0.627` and value loss dropped from `167.06` to `44.49`. However, its final all-agent evaluation reward was slightly worse than the original PPO.


## Figure 3: Original PPO vs Tuned PPO

This figure normalizes selected metrics within the Original-vs-Tuned pair. It is meant to show directionality, not absolute scale.


In [ ]:
from IPython.display import Image, display
display(Image(filename=r'C:/Users/hong/Documents/GitHub/RL_final_project/RL_final_project/final_results/1x1_ppo_tuning_comparison.png'))


## Interpretation

- **Original PPO is the strongest final safe controller** in this 1x1 evaluation. It has zero violations and the best total reward among the zero-violation agents.
- **Tuned PPO is safer than rule-based baselines and still competitive**, but its final model is slightly worse than Original PPO on total reward, average wait, and queue length.
- **Tuned PPO did improve the value function**, which suggests the hyperparameter changes helped learning quality even though the final policy did not beat the original final policy.
- **Max Pressure has the lowest average vehicle wait**, but its violation rate is above 50%, so it is not acceptable as a safe controller under this project's constraints.
- **DQN-AR and SPRe+ are safe**, but in this run they trail PPO on total reward and vehicle waiting time.

Bottom line: the original action-constrained PPO remains the best reportable 1x1 model by final all-agent evaluation reward, while the tuned PPO is useful evidence for hyperparameter sensitivity.


## File Overview

| Path | Description |
|---|---|
| `results/eval_1x1_all_agents_5ep_real_metrics_20260606_020244/` | Original all-agent 1x1 evaluation source data |
| `results/eval_1x1_tunedppo_all_agents_5ep_ppo_tuned_20260606_122740/` | Tuned PPO all-agent 1x1 evaluation source data |
| `results/ppo_1x1_real_100ep_real_metrics_20260606_020244/` | Original PPO training run |
| `results/ppo_1x1_tuned_100ep_ppo_tuned_20260606_122740/` | Tuned PPO training run |
| `final_results/1x1_metrics_comparison.png` | Figure 1 output |
| `final_results/1x1_efficiency_vs_safety.png` | Figure 2 output |
| `final_results/1x1_ppo_tuning_comparison.png` | Figure 3 output |


In [ ]:
# Reproducibility: source files used by this notebook
from pathlib import Path
import json

sources = {
    'original_eval': Path('results/eval_1x1_all_agents_5ep_real_metrics_20260606_020244/comparison.json'),
    'tuned_eval': Path('results/eval_1x1_tunedppo_all_agents_5ep_ppo_tuned_20260606_122740/comparison.json'),
    'original_ppo_summary': Path('results/ppo_1x1_real_100ep_real_metrics_20260606_020244/run_summary.json'),
    'tuned_ppo_summary': Path('results/ppo_1x1_tuned_100ep_ppo_tuned_20260606_122740/run_summary.json'),
}

for name, source in sources.items():
    print(f'{name}: {source} exists={source.exists()}')
